# 手撕Layer Normalization

In [1]:
#导入所需包
import torch
import torch.nn as nn
import torch.functional as F
import math

/home/panjiazhou/envs/pytorch/lib/python3.8/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [20]:
#编写LN网络
class Layer_Normalization(nn.Module):
    def __init__(self, LN_dim, eps=1e-5):
        super(Layer_Normalization, self).__init__()
        '''
        LN_dim 需要归一化的维度
        eps 防止除0
        '''
        self.eps = eps
        if type(LN_dim) is int:
            self.LN_dim = (LN_dim,)
        else:
            self.LN_dim = tuple(LN_dim)
        self.gamma = nn.Parameter(torch.ones(self.LN_dim))
        self.beta = nn.Parameter(torch.zeros(self.LN_dim))
    
    def forward(self,x):
        need_dim = [-(i+1) for i in range(len(self.LN_dim))]
        mean = x.mean(dim = need_dim, keepdim=True)
        var = x.var(dim = need_dim, keepdim=True, unbiased=False)
        
        result = (x-mean)/torch.sqrt(var+self.eps)

        return self.gamma*result+self.beta



In [ ]:
#编写测试用例并验证
X = torch.randn(128,16,256)
L_dim = [16,256]
l_n = Layer_Normalization(L_dim)
off_ln = nn.LayerNorm(L_dim)
off_ln.weight.data = l_n.gamma.data.clone()
off_ln.bias.data = l_n.beta.data.clone()
off_out = off_ln(X)
output = l_n(X)
print(torch.allclose(off_out,output,atol = 1e-5))


True
tensor(2.9511) tensor(-2.4589)
tensor(2.9049, grad_fn=<UnbindBackward0>) tensor(-2.4292, grad_fn=<UnbindBackward0>)


In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F

class LN(nn.Module):
    def __init__(self, dim, eps = 1e-5):
        super().__init__()
        self.dim = dim
        self.eps = eps
        self.dim = (dim,) if isinstance(dim,int) else tuple(dim)
        self.gamma = nn.Parameter(torch.ones(self.dim))
        self.beta = nn.Parameter(torch.zeros(self.dim))
    def forward(self,x):
        need_dim = [-(i+1) for i in range(len(self.dim))]
        x_mean = x.mean(dim = need_dim, keepdim=True)
        x_var = x.var(dim = need_dim, keepdim = True,unbiased = False)
        result =  (x-x_mean)/torch.sqrt(x_var+self.eps)
        return self.gamma*result+self.beta

        
X = torch.randn(128,16,256)
L_dim = [16,256]
l_n = LN(L_dim)
off_ln = nn.LayerNorm(L_dim)
off_ln.weight.data = l_n.gamma.data.clone()
off_ln.bias.data = l_n.beta.data.clone()
off_out = off_ln(X)
output = l_n(X)
print(torch.allclose(off_out,output,atol = 1e-5))

True


# 手撕RMSNorm

In [22]:
import torch.nn as nn
import torch

class RMSNorm(nn.Module):
    def __init__(self, dim):
        super(RMSNorm,self).__init__()
        self.gamma = nn.Parameter(torch.ones(dim))
        self.eps = 1e-5
        self.dim = (dim,) if isinstance(dim,int) else tuple(dim)
    def forward(self,x):
        need_dim = [-(i+1) for i in range(len(self.dim))]
        RMS = torch.sqrt(x.pow(2).mean(dim=need_dim,keepdim=True))
        return self.gamma*(x/RMS+self.eps)
X = torch.randn(128,16,256)
RN = RMSNorm(256)
out = RN(X)
print(out)

tensor([[[ 1.2660,  0.5415, -1.1315,  ...,  0.3858, -0.2284, -1.9275],
         [ 1.0406, -0.8017, -0.7683,  ..., -0.8117,  0.7364,  0.2646],
         [ 2.5883,  1.0221, -0.0673,  ...,  0.7828, -1.7501,  1.3299],
         ...,
         [-0.0920,  2.0261, -0.4451,  ...,  0.0361, -1.6372, -0.2717],
         [ 0.7297, -1.0984, -0.0162,  ..., -0.0033,  0.0974,  0.8499],
         [-1.3346,  0.7041, -0.5746,  ...,  0.2208, -1.3735,  1.5849]],

        [[-0.7041,  0.6675, -0.0318,  ...,  0.5533, -1.5625, -0.5772],
         [ 1.0155, -0.4276, -0.1726,  ...,  1.8631, -0.5108, -0.5158],
         [ 0.7796, -1.5004, -0.6387,  ..., -0.3136, -0.6186,  1.0162],
         ...,
         [ 0.9980, -0.5975,  0.2087,  ..., -1.1791, -0.2093,  0.4155],
         [ 0.7880, -0.3820, -0.7282,  ...,  0.4436, -0.4890,  0.7675],
         [ 0.9212, -0.3637,  0.8763,  ...,  0.6073,  0.4187, -1.3603]],

        [[ 0.5584, -1.5168,  0.5403,  ...,  1.3439,  0.5612,  0.1748],
         [-0.8527, -0.7013, -1.1111,  ..., -0

In [23]:
import torch.nn as nn
import torch
import math
class RMSNorm(nn.Module):
    def __init__(self,dim,eps = 1e-5):
        super().__init__()
        self.dim = (dim,) if isinstance(dim,int) else tuple(dim)
        self.eps = 1e-5
        self.gamma = nn.Parameter(torch.ones(self.dim))
        
    def forward(self,x):
        need_dim = [-(i+1) for i in range(len(self.dim))]
        rms = x.pow(2).mean(dim = need_dim, keepdim = True)
        x = x/torch.sqrt(rms+self.eps)
        return self.gamma*x
X = torch.randn(128,16,256)
RN = RMSNorm(256)
out = RN(X)
print(out)

tensor([[[ 0.6161,  0.0225,  0.8367,  ..., -0.9870, -1.1954,  1.1192],
         [-0.5966, -0.5931,  3.2581,  ..., -1.1823,  0.6077, -0.4613],
         [ 0.9129, -0.0746,  2.1917,  ...,  0.4964, -0.8853,  0.3209],
         ...,
         [ 1.7116, -0.3683,  1.3692,  ...,  1.6902,  0.0706,  0.2874],
         [-0.2749,  0.6488, -0.4924,  ..., -1.8509, -0.5326, -0.4762],
         [ 0.8109, -0.5926,  0.3558,  ...,  0.6110, -0.2428,  0.4748]],

        [[ 2.1358,  0.5806, -0.4358,  ...,  1.1411, -1.3578, -1.2472],
         [-0.2310,  2.3378, -0.6970,  ..., -1.9596, -1.2173, -2.0151],
         [ 0.3801, -0.6587, -0.9341,  ...,  0.1343, -0.2596,  0.2666],
         ...,
         [ 0.0679,  1.0808, -0.2501,  ..., -0.1397, -0.3021,  0.0651],
         [-0.1272,  0.0453,  1.1038,  ..., -0.0380, -0.6590,  1.5448],
         [-0.8632, -0.2393, -0.2103,  ...,  1.3727,  0.1998,  0.1064]],

        [[-1.9224, -0.5602, -0.5301,  ..., -0.4940, -0.4829,  0.6979],
         [ 0.0590,  1.4183,  0.0697,  ..., -0

# 手撕Batch Normalization

In [21]:
import torch.nn as nn
import torch
import math

class BatchNorm(nn.Module):
    def __init__(self, num_features):
        super(BatchNorm, self).__init__()
        self.num_features = num_features
        self.eps = 1e-5
        self.momentum = 0.1
        
        # 可学习参数
        self.gamma = nn.Parameter(torch.ones(num_features))
        self.beta = nn.Parameter(torch.zeros(num_features))
        
        # 运行时统计量
        self.register_buffer('running_mean', torch.zeros(num_features))
        self.register_buffer('running_var', torch.ones(num_features))
        self.register_buffer('num_batches_tracked', torch.tensor(0, dtype=torch.long))

    def forward(self, x):
        if self.training:
            # 沿着批次和空间维度计算统计量 (假设特征维度为最后一个维度)
            dims = list(range(x.ndim - 1))  # 例如对于(batch, seq_len, features)，dims=[0,1]
            mean = x.mean(dim=dims)
            var = x.var(dim=dims, unbiased=False)
            
            # 更新运行时统计量
            self.running_mean = (1 - self.momentum) * self.running_mean + self.momentum * mean.detach()
            self.running_var = (1 - self.momentum) * self.running_var + self.momentum * var.detach()
            self.num_batches_tracked += 1
        else:
            mean = self.running_mean
            var = self.running_var
        
        # 调整张量形状以支持广播
        shape = [1]*(x.dim()-1) + [self.num_features]
        x_norm = (x - mean.view(shape)) / torch.sqrt(var.view(shape) + self.eps)
        
        return self.gamma.view(shape) * x_norm + self.beta.view(shape)

# 验证实现正确性
X = torch.randn(128, 16, 256)
bn = BatchNorm(num_features=256)

# 创建PyTorch官方BatchNorm1d对比 (需要调整维度顺序)
ref_bn = nn.BatchNorm1d(256, eps=1e-5, momentum=0.1)
ref_bn.weight.data = bn.gamma.data.clone()
ref_bn.bias.data = bn.beta.data.clone()
ref_bn.running_mean.data = bn.running_mean.data.clone()
ref_bn.running_var.data = bn.running_var.data.clone()

# 前向计算
custom_out = bn(X)
# 官方实现需要将特征维度放到第二维
official_out = ref_bn(X.permute(0,2,1)).permute(0,2,1)

# 对比结果
flag = torch.allclose(custom_out, official_out, atol=1e-5)
print("Implementation Correct:", flag)

Implementation Correct: True
